In [1]:

from pathlib import Path
import zipfile
import shutil
import os

# ============================================================
# RESTORE ADT BACKUP — PRESERVE EXISTING FILES
# ============================================================

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "ADT_CV" / "ADT"

# Optional: put your exact backup path here when you
# have more than one attached backup.
BACKUP_PATH = None

def is_adt_zip(path):
    try:
        with zipfile.ZipFile(path) as zf:
            return "ADT_CV/ADT/run.py" in zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False

if BACKUP_PATH:
    source = Path(BACKUP_PATH)
    assert source.exists(), f"Backup not found: {source}"
else:
    candidates = []

    # ZIP archives containing the complete repository.
    for p in INPUT.rglob("*.zip"):
        if is_adt_zip(p):
            candidates.append(p)

    # Kaggle may already have extracted the ZIP.
    for manifest in INPUT.rglob("BACKUP_MANIFEST.json"):
        folder = manifest.parent
        if (folder / "ADT_CV/ADT/run.py").is_file():
            candidates.append(folder)

    if len(candidates) > 1:
        print("Multiple ADT backups detected:")
        for p in candidates:
            print(" ", p)
        raise RuntimeError(
            "Set BACKUP_PATH to the exact backup "
            "you want to restore."
        )

    if len(candidates) == 0:
        if (REPO / "run.py").is_file():
            print("ADT repository already present.")
            print("No attached complete ZIP was found.")
            print("Existing files left unchanged.")
            source = None
        else:
            raise FileNotFoundError(
                "No ADT backup found. "
                "Attach your ADT backup under Add Input."
            )
    else:
        source = candidates[0]

restored = 0
skipped = 0

def safe_destination(name):
    relative = Path(name)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe backup path: {name}")

    target = WORK / relative
    if not target.resolve().is_relative_to(WORK.resolve()):
        raise ValueError(f"Unsafe backup path: {name}")

    return target

def copy_if_missing(name, stream):
    global restored, skipped

    destination = safe_destination(name)

    if destination.exists():
        skipped += 1
        return

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(
        destination.name + ".restoring"
    )

    try:
        with temporary.open("wb") as out:
            shutil.copyfileobj(stream, out)
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)

    restored += 1

if source is not None:
    print("Restoring from:", source)

    if source.is_file():
        with zipfile.ZipFile(source) as zf:
            for item in zf.infolist():
                if item.is_dir():
                    continue
                with zf.open(item) as stream:
                    copy_if_missing(item.filename, stream)
    else:
        for path in source.rglob("*"):
            if not path.is_file() or path.is_symlink():
                continue

            relative = path.relative_to(source)
            with path.open("rb") as stream:
                copy_if_missing(str(relative), stream)

print("\n" + "=" * 65)
print("ADT RESTORE STATUS")
print("=" * 65)
print("Files restored:", restored)
print("Existing files preserved:", skipped)
print("Repository:", REPO)

assert (REPO / "run.py").is_file()
assert (REPO / "io/prepare_vogl_cv.py").is_file()

print("\nRBMA13 learned-PCEN checkpoint status:")

for fold in (1, 2, 3):
    folder = (
        REPO / "experiments/vogl_cnn_b/trainable_pcen"
        / "Architecture/rbma13_trainable_pcen"
        / "RBMA13"
        / f"fold_{fold}"
    )

    complete = all(
        (folder / name).is_file()
        for name in (
            "model.pt",
            "pcen.pt",
            "experiment_config.json",
            "test_results.json"
        )
    )

    print(f"Fold {fold}: {'COMPLETE' if complete else 'NOT COMPLETE'}")

print("\nRESTORE CHECK FINISHED")
print("Existing experiments were not overwritten.")


Restoring from: /kaggle/input/datasets/wusatullah/adt-complete-backup

ADT RESTORE STATUS
Files restored: 2739
Existing files preserved: 0
Repository: /kaggle/working/ADT_CV/ADT

RBMA13 learned-PCEN checkpoint status:
Fold 1: COMPLETE
Fold 2: NOT COMPLETE
Fold 3: NOT COMPLETE

RESTORE CHECK FINISHED
Existing experiments were not overwritten.


In [2]:
%pip install -q partitura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.9/363.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.9/261.9 kB 14.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:

from pathlib import Path
import subprocess
import sys
import json

# ============================================================
# RBMA13 — LEARNABLE PCEN — FOLD 2 ONLY
# GENERATE MEL SPECTROGRAMS
# ============================================================

REPO = Path("/kaggle/working/ADT_CV/ADT")
INPUT = Path("/kaggle/input")
PREP = REPO / "io/prepare_vogl_cv.py"

DATASET = "rbma13"
FOLD = 2

assert PREP.is_file(), f"Missing: {PREP}"

# ------------------------------------------------------------
# FIND RBMA13 AUDIO AND ANNOTATIONS
# ------------------------------------------------------------

candidates = [
    Path("/kaggle/input/datasets/wusatullah/adt-rbma/RBMA"),
    Path("/kaggle/input/adt-rbma/RBMA"),
]

RBMA_ROOT = next(
    (
        p for p in candidates
        if (p / "annotations").is_dir()
        and any(p.glob("*.wav"))
    ),
    None
)

if RBMA_ROOT is None:
    # Discover RBMA folders when the Kaggle mount name differs.
    for ann_dir in INPUT.rglob("annotations"):
        parent = ann_dir.parent
        if any(parent.glob("*.wav")):
            RBMA_ROOT = parent
            break

assert RBMA_ROOT is not None, (
    "RBMA audio and annotations not found.\n"
    "Attach your RBMA dataset using Add Input."
)

print("=" * 70)
print("RBMA13 LEARNABLE PCEN — FOLD 2 PREPROCESSING")
print("=" * 70)
print("RBMA root:", RBMA_ROOT)
print("Representation: raw Mel (PCEN learned during training)")
print("Fold: 2")

# ------------------------------------------------------------
# GENERATE MEL FEATURES USING EXISTING PREPROCESSOR
# ------------------------------------------------------------

command = [
    sys.executable,
    "-u",
    str(PREP),
    "--dataset", "rbma13",
    "--representation", "mel",
    "--fold", "2",
    "--rbma-root", str(RBMA_ROOT),
    "--output-root", str(REPO / "data/VoglCV"),
    "--seed", "42",
    "--validation-fraction", "0.15",
]

print("\nRunning:")
print(" ".join(command))
print()

subprocess.run(
    command,
    cwd=REPO,
    check=True
)

# ------------------------------------------------------------
# VERIFY FOLD 2 DATA
# ------------------------------------------------------------

FOLD_DIR = REPO / "data/VoglCV/rbma13/fold_2"

manifest_path = FOLD_DIR / "split_manifest.json"

assert manifest_path.is_file(), (
    f"Missing split manifest: {manifest_path}"
)

manifest = json.loads(manifest_path.read_text())

assert manifest["dataset"] == "rbma13"
assert manifest["fold"] == 2

print("\n" + "=" * 70)
print("VERIFYING FOLD 2 RECORDING SPECTROGRAMS")
print("=" * 70)

expected_counts = {
    "train": 15,
    "validation": 3,
    "test": 9
}

for split, expected_count in expected_counts.items():

    folder = (
        FOLD_DIR
        / "mel_recordings"
        / split
        / "rbma13"
    )

    files = sorted(folder.glob("*.pt"))

    assert len(files) == expected_count, (
        f"{split}: expected {expected_count} "
        f"recordings, found {len(files)}"
    )

    assert all(
        p.stat().st_size > 0 for p in files
    ), f"Empty feature tensor in {folder}"

    print(
        f"{split:12s}: {len(files)}/{expected_count} "
        "recordings — OK"
    )

print("\nRBMA13 FOLD 2 MEL SPECTROGRAMS READY")
print("No other folds or representations regenerated.")


RBMA13 LEARNABLE PCEN — FOLD 2 PREPROCESSING
RBMA root: /kaggle/input/datasets/wusatullah/adt-rbma/RBMA
Representation: raw Mel (PCEN learned during training)
Fold: 2

Running:
/usr/bin/python3 -u /kaggle/working/ADT_CV/ADT/io/prepare_vogl_cv.py --dataset rbma13 --representation mel --fold 2 --rbma-root /kaggle/input/datasets/wusatullah/adt-rbma/RBMA --output-root /kaggle/working/ADT_CV/ADT/data/VoglCV --seed 42 --validation-fraction 0.15

Dataset: rbma13, 27 annotated recordings, 1 folds
Fold 2 {'train': 15, 'validation': 3, 'test': 9}
rbma13 fold 2 train mel 1/15
rbma13 fold 2 train mel 2/15
rbma13 fold 2 train mel 3/15
rbma13 fold 2 train mel 4/15
rbma13 fold 2 train mel 5/15
rbma13 fold 2 train mel 6/15
rbma13 fold 2 train mel 7/15
rbma13 fold 2 train mel 8/15
rbma13 fold 2 train mel 9/15
rbma13 fold 2 train mel 10/15
rbma13 fold 2 train mel 11/15
rbma13 fold 2 train mel 12/15
rbma13 fold 2 train mel 13/15
rbma13 fold 2 train mel 14/15
rbma13 fold 2 train mel 15/15
rbma13 fold 2 va

In [4]:

from pathlib import Path
from datetime import datetime
from IPython.display import display, FileLink
import subprocess
import sys
import json
import zipfile
import hashlib
import torch

# ============================================================
# RBMA13 — LEARNABLE PCEN — FOLD 2 ONLY
# TRAIN CNN-b + AUTOMATIC BACKUP
# ============================================================

WORK = Path("/kaggle/working")
REPO = WORK / "ADT_CV" / "ADT"
RUNNER = REPO / "run.py"

DATASET = "rbma13"
FOLD = 2
REP = "trainable_pcen"

assert RUNNER.is_file(), f"Missing runner: {RUNNER}"
assert torch.cuda.is_available(), (
    "GPU not enabled. Enable GPU in Kaggle Settings."
)

DATA_DIR = REPO / "data/VoglCV/rbma13/fold_2"

for split in ("train", "validation", "test"):
    folder = DATA_DIR / "mel_recordings" / split / "rbma13"
    assert folder.is_dir() and any(folder.glob("*.pt")), (
        f"Missing {split} Mel spectrograms: {folder}"
    )

RESULT_DIR = (
    REPO / "experiments/vogl_cnn_b/trainable_pcen"
    / "Architecture/rbma13_trainable_pcen"
    / "RBMA13/fold_2"
)

required_files = [
    RESULT_DIR / "model.pt",
    RESULT_DIR / "pcen.pt",
    RESULT_DIR / "experiment_config.json",
    RESULT_DIR / "test_results.json",
]

complete = all(
    p.is_file() and p.stat().st_size > 0
    for p in required_files
)

command = [
    sys.executable, "-u", str(RUNNER),
    "--model", "vogl_cnn_b",
    "--dataset", "rbma13",
    "--fold", "2",
    "--representation", "trainable_pcen",
    "--num-samples", "1",
    "--batch-size", "8",
    "--cnn-frame-chunk", "8",
    "--num-epochs", "50",
    "--early-stop", "15",
    "--onset-tolerance-frames", "2",
    "--seed", "42",
]

if complete:
    config = json.loads(
        (RESULT_DIR / "experiment_config.json").read_text()
    )
    assert (
        config.get("representation") == REP
        and config.get("datasets") == [DATASET]
        and config.get("cv_fold") == FOLD
    ), "Existing checkpoint metadata mismatch"

    print("Fold 2 already trained. Skipping training.")

else:
    if RESULT_DIR.exists() and any(RESULT_DIR.iterdir()):
        raise RuntimeError(
            f"Partial experiment exists: {RESULT_DIR}\n"
            "Inspect before restarting to prevent overwriting."
        )

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    LOG = WORK / f"rbma13_trainable_pcen_fold2_{stamp}.log"

    print("=" * 70)
    print("RBMA13 — LEARNABLE PCEN — FOLD 2")
    print("=" * 70)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Training log:", LOG)
    print("Starting training...", flush=True)

    with LOG.open("w", encoding="utf-8") as logfile:

        process = subprocess.Popen(
            command,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
            logfile.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Training failed (exit code {return_code}). "
            f"See {LOG}"
        )

# ------------------------------------------------------------
# VERIFY TRAINED MODEL AND TEST RESULTS
# ------------------------------------------------------------

for path in required_files:
    assert path.is_file() and path.stat().st_size > 0, (
        f"Missing or empty training output: {path}"
    )

config = json.loads(
    (RESULT_DIR / "experiment_config.json").read_text()
)

assert (
    config.get("representation") == REP
    and config.get("datasets") == [DATASET]
    and config.get("cv_fold") == FOLD
), "Incorrect experiment metadata"

results = json.loads(
    (RESULT_DIR / "test_results.json").read_text()
)

micro = results["micro"]

print("\n" + "=" * 70)
print("RBMA13 FOLD 2 — FINAL TEST RESULTS")
print("=" * 70)

print(f"Precision: {100 * micro['precision']:.2f}%")
print(f"Recall:    {100 * micro['recall']:.2f}%")
print(f"Micro F1:  {100 * micro['f1']:.2f}%")

# ------------------------------------------------------------
# AUTOMATIC BACKUP IMMEDIATELY AFTER TRAINING
# ------------------------------------------------------------

BACKUP_DIR = WORK / "RBMA13_BACKUPS"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

ZIP_PATH = (
    BACKUP_DIR
    / f"RBMA13_LEARNABLE_PCEN_FOLD2_{stamp}.zip"
)

files = {}

# Save the complete fold result directory.
for path in RESULT_DIR.rglob("*"):
    if path.is_file():
        files[path.relative_to(WORK).as_posix()] = path

# Save the original fold split manifest.
manifest = DATA_DIR / "split_manifest.json"
if manifest.is_file():
    files[manifest.relative_to(WORK).as_posix()] = manifest

# Save relevant repository code for reproducibility.
for relative in (
    "run.py",
    "io/prepare_vogl_cv.py",
    "io/load.py",
    "models/trainable_pcen.py",
):
    path = REPO / relative
    if path.is_file():
        files[path.relative_to(WORK).as_posix()] = path

# Include any fold 2 training logs.
for log in WORK.glob("rbma13_trainable_pcen_fold2_*.log"):
    files[f"training_logs/{log.name}"] = log

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=4,
    allowZip64=True,
) as zf:

    for archive_name, path in sorted(files.items()):
        zf.write(path, arcname=archive_name)

    zf.writestr(
        "RBMA13_FOLD2_BACKUP_INFO.json",
        json.dumps({
            "dataset": DATASET,
            "fold": FOLD,
            "model": "vogl_cnn_b",
            "representation": REP,
            "test_micro_f1": float(micro["f1"]),
            "training_command": command,
            "created_at": stamp,
        }, indent=2),
    )

# Validate the ZIP.
with zipfile.ZipFile(ZIP_PATH) as zf:
    assert zf.testzip() is None, (
        "Backup ZIP failed its integrity test"
    )

# SHA256 integrity checksum.
sha256 = hashlib.sha256()

with ZIP_PATH.open("rb") as file:
    for block in iter(
        lambda: file.read(8 * 1024 * 1024), b""
    ):
        sha256.update(block)

CHECKSUM = ZIP_PATH.with_suffix(".zip.sha256")
CHECKSUM.write_text(
    f"{sha256.hexdigest()}  {ZIP_PATH.name}\n"
)

print("\n" + "=" * 70)
print("RBMA13 FOLD 2 TRAINING + BACKUP COMPLETED")
print("=" * 70)
print("Model:", RESULT_DIR / "model.pt")
print("Learned PCEN:", RESULT_DIR / "pcen.pt")
print("Test results:", RESULT_DIR / "test_results.json")
print("Backup ZIP:", ZIP_PATH)
print(f"Backup size: {ZIP_PATH.stat().st_size / 1024**2:.2f} MB")
print("ZIP verification: PASSED")

display(FileLink(str(ZIP_PATH)))


RBMA13 — LEARNABLE PCEN — FOLD 2
GPU: Tesla T4
Training log: /kaggle/working/rbma13_trainable_pcen_fold2_20261112_230431.log
Starting training...
Dataset files:
  Train: /kaggle/working/ADT_CV/ADT/data/VoglCV/rbma13/fold_2
  Validation: /kaggle/working/ADT_CV/ADT/data/VoglCV/rbma13/fold_2
  Test: /kaggle/working/ADT_CV/ADT/data/VoglCV/rbma13/fold_2

Using device: cuda
Model: vogl_cnn_b
Representation: trainable_pcen
Dataset: ['rbma13']
Study: Architecture
Experiment: rbma13_trainable_pcen
\nTraining dataset size: 15
Computing training-only normalization statistics...
Training mean: 0.15385743975639343
Training std: 0.22297236323356628

Initializing Ray...
2026-11-12 23:04:48,108	INFO worker.py:2024 -- Started a local Ray instance.

Starting training...
2026-11-12 23:04:48,779	INFO tensorboardx.py:43 -- pip install "ray[tune]" to see TensorBoard files.
2026-11-12 23:04:48,779	WARNING callback.py:132 -- The TensorboardX logger cannot be instantiated because either TensorboardX or one of 

/kaggle/working/RBMA13_BACKUPS/RBMA13_LEARNABLE_PCEN_FOLD2_20261113_043527_528324.zip

In [5]:

from pathlib import Path
import json
import statistics
import pandas as pd
from IPython.display import display

# ============================================================
# ENST ACCOMPANIED — VERIFY EXPERIMENTS 2, 3 AND 4
# READ-ONLY — NO TRAINING, NO FILE MODIFICATION
# ============================================================

REPO = Path("/kaggle/working/ADT_CV/ADT")
EXP_ROOT = REPO / "experiments/vogl_cnn_b"
DATA_ROOT = REPO / "data/VoglCV/enst_accompanied"

EXPERIMENTS = {
    "Experiment 2 — Smoothing only": {
        "rep": "pcen_s_only",
        "params": {
            "s": 0.018356,
            "alpha": 0.98,
            "delta": 2.0,
            "r": 0.5,
        },
    },
    "Experiment 3 — Gain only": {
        "rep": "pcen_alpha_only",
        "params": {
            "s": 0.025,
            "alpha": 0.967417,
            "delta": 2.0,
            "r": 0.5,
        },
    },
    "Experiment 4 — Smoothing + Gain": {
        "rep": "pcen_s_alpha",
        "params": {
            "s": 0.018356,
            "alpha": 0.967417,
            "delta": 2.0,
            "r": 0.5,
        },
    },
}

print("=" * 75)
print("ENST ACCOMPANIED — EXPERIMENT VERIFICATION")
print("=" * 75)
print("Repository exists:", REPO.is_dir())
print("Experiments folder exists:", EXP_ROOT.is_dir())

rows = []

for experiment, info in EXPERIMENTS.items():

    rep = info["rep"]

    result_root = (
        EXP_ROOT / rep
        / "Architecture"
        / f"enst_accompanied_{rep}"
        / "ENST_ACCOMPANIED"
    )

    print("\n" + "=" * 75)
    print(experiment)
    print("=" * 75)
    print("Expected results folder:", result_root)

    scores = []

    for fold in (1, 2, 3):

        fold_dir = result_root / f"fold_{fold}"

        model = fold_dir / "model.pt"
        config = fold_dir / "experiment_config.json"
        results = fold_dir / "test_results.json"

        model_ok = model.is_file() and model.stat().st_size > 0
        config_ok = config.is_file()
        results_ok = results.is_file()

        # Check whether the correct experiment was restored.
        config_valid = False

        if config_ok:
            try:
                cfg = json.loads(config.read_text())

                config_valid = (
                    cfg.get("representation") == rep
                    and cfg.get("datasets")
                        == ["enst_accompanied"]
                    and cfg.get("cv_fold") == fold
                )

            except (OSError, ValueError):
                config_valid = False

        # Read final test result.
        f1 = None

        if results_ok:
            try:
                result = json.loads(results.read_text())
                f1 = 100 * float(result["micro"]["f1"])

            except (OSError, ValueError, KeyError, TypeError):
                results_ok = False

        # Check feature tensors.
        data_dir = DATA_ROOT / f"fold_{fold}"

        features = {
            split: (
                data_dir
                / f"enst_accompanied_{rep}_{split}.pt"
            ).is_file()
            for split in ("train", "validation", "test")
        }

        metadata_path = (
            data_dir
            / f"enst_accompanied_{rep}_parameters.json"
        )

        params_ok = False

        if metadata_path.is_file():
            try:
                metadata = json.loads(
                    metadata_path.read_text()
                )

                params_ok = (
                    metadata.get("parameters")
                    == info["params"]
                )

            except (OSError, ValueError):
                pass

        # Classify training status.
        if (
            model_ok
            and config_valid
            and results_ok
            and f1 is not None
        ):
            status = "COMPLETE"
            scores.append(f1)

        elif fold_dir.exists() and any(fold_dir.iterdir()):
            status = "PARTIAL / CHECK"

        else:
            status = "NOT FOUND"

        print(
            f"Fold {fold}: {status:17s} | "
            f"Model: {model_ok} | "
            f"Config valid: {config_valid} | "
            f"Results: {results_ok} | "
            f"Test F1: "
            f"{f'{f1:.2f}%' if f1 is not None else 'N/A'}"
        )

        print(
            f"         Features: "
            f"Train={features['train']}, "
            f"Val={features['validation']}, "
            f"Test={features['test']} | "
            f"PCEN metadata valid={params_ok}"
        )

        rows.append({
            "Experiment": experiment,
            "Fold": fold,
            "Status": status,
            "Model": model_ok,
            "Config valid": config_valid,
            "Test results": results_ok,
            "Test F1 (%)": f1,
            "Train features": features["train"],
            "Val features": features["validation"],
            "Test features": features["test"],
            "PCEN params valid": params_ok,
        })

    if len(scores) == 3:
        print(
            f"\nMEAN TEST MICRO F1: "
            f"{statistics.mean(scores):.2f}% "
            f"± {statistics.stdev(scores):.2f} pp"
        )
    else:
        print(
            f"\nCompleted folds: {len(scores)}/3 "
            "(full experiment not verified)"
        )

# ============================================================
# FINAL SUMMARY
# ============================================================

df = pd.DataFrame(rows)

print("\n" + "=" * 75)
print("FINAL VERIFICATION SUMMARY")
print("=" * 75)

display(df)

print("\nCOMPLETED TRAINING FOLDS")
print("-" * 75)

for experiment in EXPERIMENTS:
    subset = df[df["Experiment"] == experiment]
    completed = (subset["Status"] == "COMPLETE").sum()
    print(f"{experiment}: {completed}/3")

print("\nTOTAL COMPLETED FOLDS:",
      (df["Status"] == "COMPLETE").sum(), "/9")

# ============================================================
# BACKUP DETECTION
# ============================================================

print("\n" + "=" * 75)
print("AVAILABLE BACKUPS")
print("=" * 75)

input_root = Path("/kaggle/input")

backups = [
    p for p in input_root.rglob("*")
    if p.is_file()
    and (
        "ADT_COMPLETE" in p.name.upper()
        or p.name.lower() == "backup_manifest.json"
    )
]

if backups:
    for path in backups[:20]:
        print(path)
else:
    print("No matching backup files detected under /kaggle/input.")

print("\nVERIFICATION FINISHED — NO FILES MODIFIED")


ENST ACCOMPANIED — EXPERIMENT VERIFICATION
Repository exists: True
Experiments folder exists: True

Experiment 2 — Smoothing only
Expected results folder: /kaggle/working/ADT_CV/ADT/experiments/vogl_cnn_b/pcen_s_only/Architecture/enst_accompanied_pcen_s_only/ENST_ACCOMPANIED
Fold 1: NOT FOUND         | Model: False | Config valid: False | Results: False | Test F1: N/A
         Features: Train=False, Val=False, Test=False | PCEN metadata valid=False
Fold 2: NOT FOUND         | Model: False | Config valid: False | Results: False | Test F1: N/A
         Features: Train=False, Val=False, Test=False | PCEN metadata valid=False
Fold 3: NOT FOUND         | Model: False | Config valid: False | Results: False | Test F1: N/A
         Features: Train=False, Val=False, Test=False | PCEN metadata valid=False

Completed folds: 0/3 (full experiment not verified)

Experiment 3 — Gain only
Expected results folder: /kaggle/working/ADT_CV/ADT/experiments/vogl_cnn_b/pcen_alpha_only/Architecture/enst_accom

,Experiment,Fold,Status,Model,Config valid,Test results,Test F1 (%),Train features,Val features,Test features,PCEN params valid
0,Experiment 2 — Smoothing only,1,NOT FOUND,False,False,False,None,False,False,False,False
1,Experiment 2 — Smoothing only,2,NOT FOUND,False,False,False,None,False,False,False,False
2,Experiment 2 — Smoothing only,3,NOT FOUND,False,False,False,None,False,False,False,False
3,Experiment 3 — Gain only,1,NOT FOUND,False,False,False,None,False,False,False,False
4,Experiment 3 — Gain only,2,NOT FOUND,False,False,False,None,False,False,False,False
5,Experiment 3 — Gain only,3,NOT FOUND,False,False,False,None,False,False,False,False
6,Experiment 4 — Smoothing + Gain,1,NOT FOUND,False,False,False,None,False,False,False,False
7,Experiment 4 — Smoothing + Gain,2,NOT FOUND,False,False,False,None,False,False,False,False
8,Experiment 4 — Smoothing + Gain,3,NOT FOUND,False,False,False,None,False,False,False,False



COMPLETED TRAINING FOLDS
---------------------------------------------------------------------------
Experiment 2 — Smoothing only: 0/3
Experiment 3 — Gain only: 0/3
Experiment 4 — Smoothing + Gain: 0/3

TOTAL COMPLETED FOLDS: 0 /9

AVAILABLE BACKUPS
/kaggle/input/datasets/wusatullah/adt-complete-backup/BACKUP_MANIFEST.json

VERIFICATION FINISHED — NO FILES MODIFIED
